# Mapa de selección de sitio

Dos pasos (ADR-0011):

1. **Mapa exploratorio**: dada una coordenada, dibuja la batimetría (GMRT), las isóbatas, la malla de WAVERYS, las zonas de exclusión y de colisión de usos, y hasta 3 nodos candidatos.
2. **Mapa definitivo**: con el nodo elegido y un recorte de GEBCO descargado a mano, rehace el mapa sin descargar nada.

**Antes de empezar**

1. `uv sync` en la raíz del repositorio y elegir el kernel de `.venv`.
2. `uv run copernicusmarine login` (una sola vez).
3. Indicar la carpeta de datos en `DIRECTORIO_DATOS` o con la variable de entorno `WAVE_DATA_DIR`.

## 1. Parámetros

In [ ]:
NOMBRE = "ejemplo-marcona"      # minúsculas: nombra la carpeta de salida
LATITUD = -15.40                 # grados, negativa al sur
LONGITUD = -75.30                # grados, negativa al oeste
ISOBATAS = [200, 100, 50]        # m; la primera es la principal
ISOBATA_PRINCIPAL = None         # None: la primera de ISOBATAS
SEMIANCHO_GRADOS = 1.0           # recuadro de ±semiancho alrededor de la referencia
N_CANDIDATOS = 3                 # ADR-0013: hasta 3
RESGUARDO_KM = 1.0               # ADR-0013: resguardo alrededor de las zonas de exclusión
MIN_VECINAS = 0                  # ADR-0013: las vecinas no descartan (0 a 8)
DIRECTORIO_DATOS = None          # None: variable de entorno WAVE_DATA_DIR

BATIMETRIA = {"fuente": "gmrt", "resolucion_gmrt": "med"}
WAVERYS = {"fuente": "copernicus", "fecha": "2020-07-15T12:00:00"}

## 2. Mapa exploratorio

In [ ]:
%matplotlib inline
from peruvian_wave_analysis.sitio import ConfigMapa, generar_mapa

cfg = ConfigMapa(
    nombre=NOMBRE, latitud=LATITUD, longitud=LONGITUD,
    isobatas=ISOBATAS, isobata_principal=ISOBATA_PRINCIPAL,
    semiancho_grados=SEMIANCHO_GRADOS, n_candidatos=N_CANDIDATOS, resguardo_km=RESGUARDO_KM, min_vecinas=MIN_VECINAS,
    directorio_datos=DIRECTORIO_DATOS, batimetria=BATIMETRIA, waverys=WAVERYS,
)
res = generar_mapa(cfg)
res.figura

## 3. Capas de zonas

Una capa que no se pudo descargar aparece como «no evaluada»: no significa que no haya zonas.

In [ ]:
import pandas as pd

pd.DataFrame(res.zonas.capas).T[["titulo", "estado", "n_poligonos"]]

## 4. Candidatos

Nodos que cumplen los criterios de ADR-0013 (nodo con dato, celda sin tierra, fuera de las zonas de exclusión), ordenados por distancia a la isóbata principal. El número coincide con el del mapa. Las colisiones de uso se declaran, no descartan.

In [ ]:
principal = f"dist_iso_{cfg.isobata_principal:g}m_km"
columnas = ["candidato", "lat", "lon", "prof_nodo_m", "prof_media_celda_m", principal,
            "dist_costa_km", "dist_exclusion_km", "zonas_colision"]
res.tabla[res.tabla["candidato"].notna()][columnas].round(2)

Motivo de descarte de los demás nodos:

In [ ]:
res.tabla["motivo_descarte"].replace("", "cumple").value_counts()

## 5. Siguiente paso: GEBCO

Descargar el recorte indicado en https://download.gebco.net/ (formato NetCDF).

In [ ]:
from IPython.display import Markdown

Markdown(res.salidas["siguiente_paso"].read_text(encoding="utf-8"))

## 6. Mapa definitivo

Completar cuando ya se haya descargado GEBCO y elegido el nodo. No descarga nada: usa los insumos de la corrida exploratoria y verifica su hash. Si el nodo no cumple los criterios con GEBCO, falla y borra la salida definitiva anterior.

In [ ]:
ARCHIVO_GEBCO_DESCARGADO = None   # p. ej. r"C:\Descargas\gebco_2025_recorte.nc"
VERSION_GEBCO = "GEBCO_2025"
NODO_LAT, NODO_LON = None, None    # coordenadas exactas del nodo elegido (columnas lat y lon)

if ARCHIVO_GEBCO_DESCARGADO and NODO_LAT is not None:
    from peruvian_wave_analysis import rutas
    from peruvian_wave_analysis.sitio import gebco
    from peruvian_wave_analysis.sitio.final import ConfigFinal, generar_mapa_final

    ruta_gebco, _ = gebco.registrar(ARCHIVO_GEBCO_DESCARGADO, rutas.directorio_datos(DIRECTORIO_DATOS), VERSION_GEBCO)
    final = generar_mapa_final(ConfigFinal(exploratorio=str(res.salidas["metadatos"]), gebco=str(ruta_gebco),
                                           nodo_lat=NODO_LAT, nodo_lon=NODO_LON))
    display(final.figura)
    print(final.salidas)
else:
    print("Pendiente: descargar GEBCO y elegir el nodo.")

## Límites

- Sin fuente en línea para zonas militares, tráfico marítimo, fondeaderos, cables, franja de 5 millas, caladeros ni acuicultura.
- Los servicios de zonas no declaran su fecha de actualización; se usa la fecha de acceso.
- Cerca de la costa la batimetría global interpola entre sondeos; WAVERYS usa su propia batimetría, más gruesa.
- El nodo representa una celda de 0,2° (unos 22 km): las zonas se evalúan sobre la celda completa.